In [4]:
pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.1/8.3 MB 2.0 MB/s eta 0:00:05
   - -------------------------------------- 0.3/8.3 MB 3.9 MB/s eta 0:00:03
   ---- ----------------------------------- 1.0/8.3 MB 7.6 MB/s eta 0:00:01
   ------- -------------------------------- 1.6/8.3 MB 9.2 MB/s eta 0:00:01
   ---------- ----------------------------- 2.1/8.3 MB 9.6 MB/s eta 0:00:01
   ------------- -------------------------- 2.8/8.3 MB 10.5 MB/s eta 0:00:01
   ----------------- ---------------------- 3.7/8.3 MB 11.7 MB/s eta 0:00:01
   ---------------------- ----------------- 4.6/8.3 MB 12.7 MB/s eta 0:00:01
   ----------------------- ---------------- 4.9/8.3 MB 13.0 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.3 MB 11.4 MB/s eta 0:00:01
   ------------------------- -------------- 5.2/8.3 MB 10.7 MB/s eta 0:00:01
   -------------------------- ------------- 5.4/8.3 MB 9.9 MB/s eta 0:00:01
   ----------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
print('hey')
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
print('All Set')

hey
All Set


In [10]:
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

'wget' is not recognized as an internal or external command,
operable program or batch file.
'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [17]:
import os
import zipfile

zip_file = "book-crossings.zip"

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(".")

books_filename = "BX-Books.csv"
ratings_filename = "BX-Book-Ratings.csv"

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [18]:
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [24]:
print([name for name in globals() if not name.startswith('_')])
print(df_books.head())
print(df_ratings.head())

['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'np', 'pd', 'csr_matrix', 'NearestNeighbors', 'plt', 'books_filename', 'ratings_filename', 'os', 'zipfile', 'urllib', 'url', 'zip_file', 'zip_ref', 'df_books', 'df_ratings']
         isbn                                              title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

                 author  
0    Mark P. O. Morford  
1  Richard Bruce Wright  
2          Carlo D'Este  
3      Gina Bari Kolata  
4       E. J. W. Barber  
     user        isbn  rating
0  276725  034545104X     0.0
1  276726  0155061224     5.0
2  276727  0446520802     0.0
3  276729  052165615X     3.0
4  276729  0521795028     6.0


In [34]:
user_counts = df_ratings.groupby('user')['rating'].count()
book_counts = df_ratings.groupby('isbn')['rating'].count()

df_ratings['user_rating_count'] = df_ratings['user'].map(user_counts)
df_ratings['book_rating_count'] = df_ratings['isbn'].map(book_counts)

df_ratings = df_ratings[
    (df_ratings['user_rating_count'] >= 200) &
    (df_ratings['book_rating_count'] >= 100)
]

df_ratings = df_ratings.drop(
    columns=['user_rating_count', 'book_rating_count']
)

print("Ratings after filtering:", len(df_ratings))
print("Users:", df_ratings['user'].nunique())
print("Books:", df_ratings['isbn'].nunique())
print("Ratings after filtering:", len(df_ratings))
print("Users:", df_ratings['user'].nunique())
print("Books:", df_ratings['isbn'].nunique())

Ratings after filtering: 1244
Users: 21
Books: 100
Ratings after filtering: 1244
Users: 21
Books: 100


In [28]:
df = df_ratings.pivot_table(
    index='isbn',
    columns='user',
    values='rating'
).fillna(0)

print(df.shape)

(731, 888)


In [27]:
books = df_books[df_books['isbn'].isin(df.index)].copy()

print("Books in matrix:", len(books))
print(books.head())

Books in matrix: 727
          isbn                                 title            author
18  0440234743                         The Testament      John Grisham
19  0452264464  Beloved (Plume Contemporary Fiction)     Toni Morrison
26  0971880107                           Wild Animus      Rich Shapero
27  0345402871                              Airframe  Michael Crichton
28  0345417623                              Timeline  MICHAEL CRICHTON


In [29]:
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

model.fit(df)
print("KNN model trained successfully!")

KNN model trained successfully!


In [53]:
def get_recommends(title=""):
    book = good_books[good_books["title"] == title]

    if book.empty:
        return [title, []]

    b = df_book_features.loc[
        df_book_features.index.isin(book["isbn"])
    ]

    distances, indices = model.kneighbors(
        b.values,
        n_neighbors=6
    )

    distances = distances[0][1:]
    indices = indices[0][1:]

    titles = [
        df_books.loc[
            df_books["isbn"] == df_book_features.iloc[i].name,
            "title"
        ].values[0]
        for i in indices
    ]

    recommended = [
        list(z) for z in zip(titles, distances)
    ][::-1]

    return [title, recommended]

print("Correct get_recommends function loaded!")

Correct get_recommends function loaded!


In [31]:
result = get_recommends(
    "The Queen of the Damned (Vampire Chronicles (Paperback))"
)

print(result)

['The Queen of the Damned (Vampire Chronicles (Paperback))', [['The Vampire Lestat (Vampire Chronicles, Book II)', 0.4821588397026062], ['The Tale of the Body Thief (Vampire Chronicles (Paperback))', 0.46236616373062134], ['Interview with the Vampire', 0.26549315452575684], ['The Witching Hour (Lives of the Mayfair Witches)', 0.25513434410095215], ['Catch 22', 0.20601648092269897]]]


In [32]:
target = "The Queen of the Damned (Vampire Chronicles (Paperback))"

print(df_books[df_books['title'] == target])

             isbn                                              title  \
2527   0345351525  The Queen of the Damned (Vampire Chronicles (P...   
72840  0833563505  The Queen of the Damned (Vampire Chronicles (P...   

          author  
2527   Anne Rice  
72840  Anne Rice  


In [33]:
target_isbns = df_books[df_books['title'] == target]['isbn']

print(df_ratings[df_ratings['isbn'].isin(target_isbns)]['isbn'].value_counts())

isbn
0345351525    95
Name: count, dtype: int64


In [35]:
df_ratings = pd.read_csv(
    ratings_filename,
    sep=';',
    encoding='latin-1',
    on_bad_lines='skip'
)

print("Original ratings:", len(df_ratings))

Original ratings: 1149780


In [37]:
print(df_ratings.columns)
print(df_ratings.head())


Index(['User-ID', 'ISBN', 'Book-Rating'], dtype='object')
   User-ID        ISBN  Book-Rating
0   276725  034545104X            0
1   276726  0155061224            5
2   276727  0446520802            0
3   276729  052165615X            3
4   276729  0521795028            6


In [40]:
df_ratings = df_ratings.rename(columns={
    'User-ID': 'user',
    'ISBN': 'isbn',
    'Book-Rating': 'rating'
})
print(df_ratings.columns)

Index(['user', 'isbn', 'rating'], dtype='object')


In [41]:
user_counts = df_ratings.groupby('user')['rating'].count()
book_counts = df_ratings.groupby('isbn')['rating'].count()

df_ratings['user_rating_count'] = df_ratings['user'].map(user_counts)
df_ratings['book_rating_count'] = df_ratings['isbn'].map(book_counts)

df_ratings = df_ratings[
    (df_ratings['user_rating_count'] >= 200) &
    (df_ratings['book_rating_count'] >= 100)
]

df_ratings = df_ratings.drop(
    columns=['user_rating_count', 'book_rating_count']
)

print("Ratings after filtering:", len(df_ratings))
print("Users:", df_ratings['user'].nunique())
print("Books:", df_ratings['isbn'].nunique())

Ratings after filtering: 49781
Users: 888
Books: 731


In [42]:
good_books = pd.merge(
    df_ratings,
    df_books,
    on='isbn'
)

good_books = good_books.drop_duplicates(
    ['title', 'user']
)

print("Good books shape:", good_books.shape)

Good books shape: (49136, 5)


In [43]:
df_book_features = good_books.pivot(
    index='isbn',
    columns='user',
    values='rating'
).fillna(0)

print("Matrix shape:", df_book_features.shape)

Matrix shape: (727, 888)


In [44]:
mat_book_features = csr_matrix(df_book_features.values)

model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

model.fit(mat_book_features)

print("KNN model trained successfully!")

KNN model trained successfully!


In [45]:
def get_recommends(title=""):
    book = good_books[good_books["title"] == title]

    if book.empty:
        return [title, []]

    b = df_book_features.loc[
        df_book_features.index.isin(book["isbn"])
    ]

    distances, indices = model.kneighbors(
        b.values,
        n_neighbors=6
    )

    distances = distances[0][1:]
    indices = indices[0][1:]

    titles = [
        df_books.loc[
            df_books["isbn"] == df_book_features.iloc[i].name,
            "title"
        ].values[0]
        for i in indices
    ]

    recommended = [
        list(z) for z in zip(titles, distances)
    ][::-1]

    return [title, recommended]

print("Function created successfully!")

Function created successfully!


In [46]:
results = get_recommends(
    "The Queen of the Damned (Vampire Chronicles (Paperback))"

)
print(result)

['The Queen of the Damned (Vampire Chronicles (Paperback))', [['The Vampire Lestat (Vampire Chronicles, Book II)', 0.4821588397026062], ['The Tale of the Body Thief (Vampire Chronicles (Paperback))', 0.46236616373062134], ['Interview with the Vampire', 0.26549315452575684], ['The Witching Hour (Lives of the Mayfair Witches)', 0.25513434410095215], ['Catch 22', 0.20601648092269897]]]


In [54]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2): 
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", np.float64(0.8016210581447822)], ['The Weight of Water', np.float64(0.7708583572697412)], ['The Surgeon', np.float64(0.7699410973804288)], ['I Know This Much Is True', np.float64(0.7677075092617776)], ['The Lovely Bones: A Novel', np.float64(0.7234864549790632)]]]
You passed the challenge! 🎉🎉🎉🎉🎉
